In [ ]:
# Cell 1 — check GPU
!nvidia-smi

In [ ]:
# Cell 2 — install pinned deps (keep Kaggle's CUDA torch build; the actual torch version is recorded in config.json)
import glob, os
IN = os.path.dirname(glob.glob("/kaggle/input/**/train_simcse.py", recursive=True)[0])
print("dataset dir:", IN)
!grep -v '^torch==' {IN}/requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

In [ ]:
# Cell 3 — recreate the repo layout under /kaggle/working so paths resolve unmodified
import shutil
W = "/kaggle/working"
layout = {"train_simcse.py": "src/train_simcse.py", "evaluate.py": "src/evaluate.py",
          "ablation_unsup_same_mask.json": "configs/ablation_unsup_same_mask.json",
          "unsup_sentences.txt": "data/processed/unsup_sentences.txt",
          "stsb_dev.jsonl": "data/stsb/dev.jsonl", "requirements.txt": "requirements.txt"}
for src, dst in layout.items():
    os.makedirs(os.path.join(W, os.path.dirname(dst)), exist_ok=True)
    shutil.copy(os.path.join(IN, src), os.path.join(W, dst))
os.chdir(W)
!cd {IN} && sha256sum -c SHA256SUMS
!find src configs data -type f | sort

In [ ]:
# Cell 4 — train (Part 5 ablation: Mode A with same_dropout_mask=true, both views share one dropout mask)
RUN_ID = "ablation_unsup_same_mask"
HW = f"Kaggle {torch.cuda.get_device_name(0).replace('Tesla ', '')} x{torch.cuda.device_count()}"
print(HW)
!cd /kaggle/working && python src/train_simcse.py --mode unsup --config configs/ablation_unsup_same_mask.json --output_dir runs/{RUN_ID} --hardware "{HW}"

In [ ]:
# Cell 5 — verify run_record.json (written by the script) and trim the output
import json
rec = json.load(open(f"/kaggle/working/runs/{RUN_ID}/run_record.json"))
need = {"run_id","mode","timestamp","config","seed","hardware","results","checkpoint_path","notes"}
assert need <= rec.keys(), need - rec.keys()
assert rec["mode"] == "unsupervised" and rec["config"]["same_dropout_mask"] is True
assert rec["results"]["dev_spearman"] is not None and rec["results"]["test_spearman"] is None
print(json.dumps({k: rec[k] for k in ("run_id","hardware","results","notes")}, indent=2))
shutil.rmtree("/kaggle/working/data")   # inputs, no need to re-download them
!find /kaggle/working/runs -type f | sort && du -sh /kaggle/working/runs